In [3]:
from youtube_transcript_api import YouTubeTranscriptApi
from agents import Agent, Runner
from dotenv import load_dotenv
import os
import yt_dlp
from typing import Any
import requests


# Experimenting with FPL API

In [9]:
from __future__ import annotations

import requests
from typing import Any


class FPLService:
    BASE_URL = "https://fantasy.premierleague.com/api"

    def __init__(self, timeout: int = 20) -> None:
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": "fpl-agent/0.1",
                "Accept": "application/json",
            }
        )

    def _get(self, path: str) -> Any:
        url = f"{self.BASE_URL}{path}"
        response = self.session.get(url, timeout=self.timeout)
        response.raise_for_status()
        return response.json()

    def get_bootstrap_static(self) -> dict[str, Any]:
        return self._get("/bootstrap-static/")

    def get_fixtures(self) -> list[dict[str, Any]]:
        return self._get("/fixtures/")

    def get_event_live(self, gameweek: int) -> dict[str, Any]:
        return self._get(f"/event/{gameweek}/live/")

    def get_element_summary(self, player_id: int) -> dict[str, Any]:
        return self._get(f"/element-summary/{player_id}/")

    def get_entry(self, team_id: int) -> dict[str, Any]:
        return self._get(f"/entry/{team_id}/")

    def get_entry_picks(self, team_id: int, gameweek: int) -> dict[str, Any]:
        return self._get(f"/entry/{team_id}/event/{gameweek}/picks/")

    def get_my_team(self, team_id: int, gameweek: int) -> dict[str, Any]:
        return self._get(f"/entry/{team_id}/event/{gameweek}/picks/")

    def get_player_news(self) -> list[dict[str, Any]]:
        data = self.get_bootstrap_static()
        players = data.get("elements", [])
        teams = {team["id"]: team["name"] for team in data.get("teams", [])}

        news_items: list[dict[str, Any]] = []

        for player in players:
            news_text = (player.get("news") or "").strip()
            if not news_text:
                continue

            news_items.append(
                {
                    "player_id": player["id"],
                    "name": player["web_name"],
                    "team_id": player["team"],
                    "team_name": teams.get(player["team"]),
                    "status": player.get("status"),
                    "chance_of_playing_next_round": player.get("chance_of_playing_next_round"),
                    "chance_of_playing_this_round": player.get("chance_of_playing_this_round"),
                    "news": news_text,
                }
            )

        return news_items

In [7]:
import pandas as pd

fpl = FPLService()
bootstrap = fpl.get_bootstrap_static()

players_df = pd.DataFrame(bootstrap["elements"])[
    [
        "id",
        "first_name",
        "second_name",
        "web_name",
        "team",
        "element_type",
        "now_cost",
        "total_points",
        "minutes",
        "goals_scored",
        "assists",
        "selected_by_percent",
    ]
]

teams_df = pd.DataFrame(bootstrap["teams"])[
    [
        "id",
        "name",
        "short_name",
        "strength",
        "strength_attack_home",
        "strength_attack_away",
        "strength_defence_home",
        "strength_defence_away",
    ]
]

gameweeks_df = pd.DataFrame(bootstrap["events"])[
    [
        "id",
        "name",
        "deadline_time",
        "finished",
        "is_current",
        "is_next",
    ]
]


In [8]:
players_df.head(50)


,id,first_name,second_name,web_name,team,element_type,now_cost,total_points,minutes,goals_scored,assists,selected_by_percent
0,1,David,Raya Martín,Raya,1,1,60,129,2790,0,0,34.6
1,2,Kepa,Arrizabalaga Revuelta,Arrizabalaga,1,1,40,0,0,0,0,0.4
2,3,Karl,Hein,Hein,1,1,40,0,0,0,0,0.2
3,4,Tommy,Setford,Setford,1,1,39,0,0,0,0,0.2
4,5,Gabriel,dos Santos Magalhães,Gabriel,1,2,72,173,2165,3,4,43.0
5,6,William,Saliba,Saliba,1,2,61,107,2074,1,0,13.2
6,7,Riccardo,Calafiori,Calafiori,1,2,56,93,1446,1,2,6.0
7,8,Jurriën,Timber,J.Timber,1,2,63,149,2452,3,6,26.8
8,9,Jakub,Kiwior,Kiwior,1,2,54,0,0,0,0,0.0
9,10,Myles,Lewis-Skelly,Lewis-Skelly,1,2,50,12,311,0,0,1.4
